# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('C:\\Users\\SAM\\OneDrive\\Documents\\DV\\data-viz-class-material\\data\\global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
import plotly.express as px

# 1. Filter to fossil sources only
# (Using the hint provided in your notebook, assuming 'df' is your main DataFrame)
df_fossil = df[df["Source_Type"] == "Fossil"].copy()

# 2. Build the Treemap
fig = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],  # Hierarchical breakdown
    values="TWh",  # Size of rectangles determined by TWh
    color="Source",  # Color encodes the fossil source type
    # Color map using a CVD-safe (Color Vision Deficiency) palette (e.g., Okabe-Ito variations)
    color_discrete_map={
        "Coal": "#E69F00",  # Orange
        "Oil": "#56B4E9",  # Sky Blue
        "Natural Gas": "#009E73",  # Bluish Green
    },
    title="Europe is the Most Fossil-Dependent Region (Example Insight Title)",  # Change based on your actual data insight
)

# 3. Requirements for Labels and Node Colors
# - Show TWh values only (no percentages)
# - Grey out parent nodes (Region and Country level)
fig.update_traces(
    textinfo="label+value",  # Displays the name and raw value, omits percentages
    # Setting the background of parent levels to grey while letting leaf nodes use the discrete map
    marker=dict(depthfade="reversed"),
)

# Display the plot
fig.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:
import plotly.express as px

# 1. Load tips dataset
df = px.data.tips()

# 2. Build the Sunburst chart with requirements
fig = px.sunburst(
    df,
    path=['day', 'time', 'smoker'],       # Hierarchy
    values='total_bill',                  # Aggregates sum of total bill automatically
    color='smoker',                       # Color encodes smoker status
    color_discrete_map={                  # CVD-safe blue/orange palette (Okabe-Ito)
        'Yes': '#E69F00',                 # Orange
        'No': '#56B4E9'                   # Sky Blue
    },
    title="<b>Where Does the Most Spending Happen?</b><br>Dinner on Saturdays accounts for the largest share of total bills."
)

# 3. Use percent parent for text labels
fig.update_traces(
    textinfo="label+percent parent",
    insidetextorientation="radial"
)

# 4. Optional: Explicitly style parent levels to be greyed out 
# Plotly colors branches based on the leaf node colors by default, 
# but we can fine-tune the layout or markers if strict greying of root/parent is needed.
fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

# Display the chart
fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [5]:

import pandas as pd
import plotly.express as px

# 1. Load the dataset
df = pd.read_csv("C:\\Users\\SAM\\OneDrive\\Documents\\DV\\data-viz-class-material\\data\\global_energy_mix.csv")

# 2. Filter to Nuclear & Hydro and create the 'Source_Type' classification as requested
# Requirement: "Low-carbon energy (Nuclear + Hydro)"
low_carbon_sources = ['Nuclear', 'Hydro']
filtered_df = df[df['Source'].isin(low_carbon_sources)].copy()
filtered_df['Source_Type'] = 'Low-carbon'

# Aggregate TWh by country
agg_df = filtered_df.groupby('Country', as_index=False)['TWh'].sum()

# Treemap Requirement: single-level path=['All', 'Country'] with dummy root 'Low-carbon'
agg_df['All'] = 'Low-carbon'

# Bar Chart Requirements: sorted by TWh, horizontal orientation, find leading country
agg_df_sorted = agg_df.sort_values(by='TWh', ascending=True)
leading_country = agg_df.sort_values(by='TWh', ascending=False).iloc[0]['Country']

# CVD-safe color palette choice (Okabe-Ito Sky Blue)
cvd_safe_blue = '#56B4E9'


# --- TREEMAP CHART ---
fig_tree = px.treemap(
    agg_df,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=[cvd_safe_blue],
    title="<b>Treemap: Share of Low-Carbon Energy (Nuclear + Hydro) by Country</b>"
)
fig_tree.show()


# --- HORIZONTAL BAR CHART ---
fig_bar = px.bar(
    agg_df_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=[cvd_safe_blue],
    title=f"<b>{leading_country} Leads in Low-Carbon Energy Generation</b><br>Total Nuclear + Hydro generation in TWh"
)

# Ensure the categorical order is strictly preserved from the sorted dataframe
fig_bar.update_layout(yaxis={'type': 'category'})
fig_bar.show()